# Predicción de Precios de Vivienda
### Modelo de Regresión Lineal con Scikit-Learn

**Autor:** Alcides  
**Universidad:** University of the People — Business Analytics & Data Science  
**Fecha:** Junio 2026  

---

## Objetivo
Construir y evaluar un modelo de Machine Learning para predecir precios de vivienda, 
demostrando un pipeline completo: carga de datos → EDA → preprocesamiento → modelado → evaluación comparativa.

**Variable objetivo:** `price` (precio en USD)  
**Algoritmos:** Regresión Lineal · Decision Tree · Random Forest


## 1. Configuración del Entorno

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Estilo visual profesional
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.facecolor': '#F8F8F8', 'figure.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.titlesize': 12, 'axes.titleweight': '500',
})
C1, C2, C3 = '#185FA5', '#0F6E56', '#BA7517'

print("✓ Librerías cargadas correctamente")

## 2. Carga y Exploración de Datos

In [ ]:
df = pd.read_csv("house_prices.csv")

print(f"Dimensiones del dataset: {df.shape}")
print(f"Registros: {df.shape[0]:,} | Variables: {df.shape[1]}")
df.head()

In [ ]:
# Información del dataset
df.info()

In [ ]:
# Estadísticas descriptivas
df.describe().round(2)

## 3. Análisis Exploratorio (EDA)

### 3.1 Distribución de la variable objetivo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Distribución del precio de viviendas', fontsize=13, fontweight='500')

axes[0].hist(df['price']/1e6, bins=50, color=C1, alpha=0.8, edgecolor='white', lw=0.4)
axes[0].set_xlabel('Precio (millones USD)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución original (asimétrica)')
med = df['price'].median()/1e6
axes[0].axvline(med, color=C3, lw=2, linestyle='--', label=f'Mediana: ${med:.2f}M')
axes[0].legend()

axes[1].hist(np.log1p(df['price']), bins=50, color=C2, alpha=0.8, edgecolor='white', lw=0.4)
axes[1].set_xlabel('Log(Precio)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Transformación logarítmica (más normal)')

plt.tight_layout()
plt.show()
print(f"Precio mínimo:  ${df['price'].min():>12,.0f}")
print(f"Precio mediano: ${df['price'].median():>12,.0f}")
print(f"Precio máximo:  ${df['price'].max():>12,.0f}")

### 3.2 Mapa de correlaciones

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
cols = ['price','sqft_living','bedrooms','bathrooms','floors','condition','yr_built']
corr = df[cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            annot=True, fmt='.2f', square=True, linewidths=.5, ax=ax,
            cbar_kws={'shrink': .6}, annot_kws={'size': 9})
ax.set_title('Correlaciones entre variables numéricas')
plt.tight_layout()
plt.show()

print("Variable con mayor correlación con el precio:")
print(corr['price'].drop('price').abs().sort_values(ascending=False))

### 3.3 Relación entre características y precio

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Relación entre características y precio', fontsize=13, fontweight='500')

feats = [('sqft_living','Superficie habitable (pie²)'),
         ('bedrooms','Habitaciones'), ('bathrooms','Baños'), ('floors','Pisos')]

for ax, (col, label) in zip(axes.flatten(), feats):
    ax.scatter(df[col], df['price']/1e6, alpha=0.15, s=8, color=C1)
    m_c, b_c = np.polyfit(df[col], df['price']/1e6, 1)
    xs = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(xs, m_c*xs+b_c, color=C3, lw=2, label='Tendencia')
    ax.set_xlabel(label); ax.set_ylabel('Precio (M USD)'); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

## 4. Preprocesamiento de Datos

### 4.1 Valores faltantes

In [ ]:
print("Valores faltantes por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Rellenar con la mediana (resistente a valores atípicos)
mediana_condition = df['condition'].median()
df['condition'] = df['condition'].fillna(mediana_condition)

print(f"\n✓ Mediana usada para 'condition': {mediana_condition}")
print(f"✓ Valores faltantes restantes: {df.isnull().sum().sum()}")
df.head()

### 4.2 Selección de características y división de datos

In [ ]:
# Características seleccionadas y variable objetivo
X = df[['sqft_living', 'bedrooms', 'bathrooms', 'floors']]
y = df['price']

print(f"Shape de X (características): {X.shape}")
print(f"Shape de y (objetivo):        {y.shape}")

# División 80% entrenamiento / 20% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"\nEntrenamiento: {X_train.shape[0]:,} registros ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Prueba:         {X_test.shape[0]:,} registros  ({X_test.shape[0]/len(X)*100:.0f}%)")

## 5. Modelos de Machine Learning

### 5.1 Regresión Lineal (modelo base)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred_lr = model.predict(X_test)

print("Coeficientes aprendidos:")
for feat, coef in zip(X.columns, model.coef_):
    print(f"  {feat:>15}: ${coef:>10,.2f}")
print(f"  {'Intercepto':>15}: ${model.intercept_:>10,.2f}")

### 5.2 Decision Tree y Random Forest

In [ ]:
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)
y_pred_tr = tree.predict(X_test)

forest = RandomForestRegressor(random_state=42)
forest.fit(X_train, y_train)
y_pred_rf = forest.predict(X_test)

print("✓ Modelos entrenados: Decision Tree y Random Forest")

## 6. Evaluación y Comparación

In [ ]:
results = {
    'Regresión Lineal': (y_pred_lr, C1),
    'Decision Tree':    (y_pred_tr, C2),
    'Random Forest':    (y_pred_rf, C3),
}

print(f"{'Modelo':<20} {'MSE':>20} {'R²':>8} {'MAE':>14}")
print("-" * 66)
for name, (pred, _) in results.items():
    mse_v = mean_squared_error(y_test, pred)
    r2_v  = r2_score(y_test, pred)
    mae_v = mean_absolute_error(y_test, pred)
    print(f"{name:<20} {mse_v:>20,.0f} {r2_v:>8.4f} {mae_v:>14,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Comparación de modelos', fontsize=13, fontweight='500')

names  = list(results.keys())
colors_m = [C1, C2, C3]

for ax, metric, title in zip(axes,
    ['r2','mae','mse'],
    ['R² (mayor es mejor)','MAE en USD','MSE']):
    vals = [
        r2_score(y_test, results[n][0]) if metric=='r2' else
        mean_absolute_error(y_test, results[n][0]) if metric=='mae' else
        mean_squared_error(y_test, results[n][0])
        for n in names
    ]
    bars = ax.bar(names, vals, color=colors_m, alpha=0.85, width=0.5, edgecolor='white')
    ax.set_title(title, fontsize=10)
    for bar, val in zip(bars, vals):
        fmt = '.3f' if metric=='r2' else ',.0f'
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                format(val, fmt), ha='center', va='bottom', fontsize=8, fontweight='500')
    ax.tick_params(axis='x', labelsize=8)
plt.tight_layout(); plt.show()

### 6.1 Diagnóstico del modelo lineal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Regresión Lineal — Diagnóstico', fontsize=13, fontweight='500')

ax = axes[0]
ax.scatter(y_test/1e6, y_pred_lr/1e6, alpha=0.2, s=8, color=C1)
lim = max(y_test.max(), y_pred_lr.max())/1e6
ax.plot([0,lim],[0,lim], color=C3, lw=2, linestyle='--', label='Predicción perfecta')
ax.set_xlabel('Precio real (M USD)'); ax.set_ylabel('Precio predicho (M USD)')
ax.set_title('Real vs Predicho'); ax.legend()

residuals = y_test - y_pred_lr
ax = axes[1]
ax.hist(residuals/1e3, bins=50, color=C2, alpha=0.8, edgecolor='white', lw=0.4)
ax.axvline(0, color=C3, lw=2, linestyle='--', label='Residuo = 0')
ax.set_xlabel('Residuo (miles USD)'); ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de residuos'); ax.legend()

plt.tight_layout(); plt.show()

print(f"Media residuos:          ${residuals.mean():>10,.2f}")
print(f"Desv. estándar residuos: ${residuals.std():>10,.2f}")

### 6.2 Importancia de variables — Random Forest

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
importances = pd.Series(forest.feature_importances_, index=X.columns).sort_values()
colors_imp = [C1 if v == importances.max() else '#888780' for v in importances.values]
bars = ax.barh(importances.index, importances.values, color=colors_imp, alpha=0.85, edgecolor='white')
ax.set_xlabel('Importancia relativa')
ax.set_title('Importancia de variables — Random Forest')
for bar, val in zip(bars, importances.values):
    ax.text(val+0.003, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

## 7. Conclusiones

| Hallazgo | Detalle |
|---|---|
| **Mejor modelo** | Random Forest supera a la Regresión Lineal en todas las métricas |
| **Variable clave** | `sqft_living` es el predictor más importante del precio |
| **Baseline sólido** | La Regresión Lineal con R²≈0.29 es un punto de partida útil |
| **Residuos sin sesgo** | Media de residuos ≈ $0, indicando predicciones no sesgadas |

### Próximos pasos
- Incorporar variables de localización (coordenadas, barrio)
- Aplicar transformación log al precio para mejorar la normalidad
- Ajustar hiperparámetros con `GridSearchCV`
- Explorar modelos de gradient boosting (XGBoost, LightGBM)

---
*Proyecto de portafolio — Alcides · University of the People · 2026*
